# LECTURA DE LAS 3 TABLAS

In [63]:
import re
import pandas as pd
import pdfplumber
import os
from tqdm import tqdm
import numpy as np

In [40]:
pdf_folder = "../pdfs"

# Lista de archivos PDF
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
pdf_files

['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf',
 '16_enero_2026.pdf',
 '16_setiembre_2025.pdf',
 '17_abril_2026.pdf',
 '17_diciembre_2025.pdf',
 '17_febrero_2026.pdf',
 '17_noviembre_2025.pdf',
 '17_setiembre_2025.pdf',
 '18_diciembre_2025.pdf',
 '18_febrero_2026.pdf',
 '18_marzo_2026.pdf',
 '18_noviembre_2025.pdf',
 '18_setiembre_2025.pdf',
 '19_diciembre_2025.pdf',
 '19_enero_2026.pdf',
 '19_febrero_2026.pdf',
 '19_marzo_2026.pdf',
 '19_noviembre_2025.pdf',
 '19_setiembre_2025.pdf',
 '20_abril_2026.pdf',
 '20_enero_2026.pdf',
 '20_febrero_2026.pdf',
 '20_marzo_2026.pdf',
 '20_noviembre_2025.pdf',
 '20_octubre_2025.pdf',
 '21_abril_2026.pdf',
 '21_enero_2026.pdf',
 '21_noviembre_2025.pdf',
 '21_octubre_2025.pdf',
 '22_abril_2026.pdf',
 '22_diciembre_2025.pdf',
 '22_enero_2026.pdf',
 '22_octubre_2025.pd

# TABLA 1

## PRECIOS DE HUEVO SEGUN TIPO DE COMERCIALIZACION (Soles por kilogramo)


In [41]:
# mapeo de meses
meses_map = {
    'ene':1, 'feb':2, 'mar':3, 'abr':4, 'may':5, 'jun':6,
    'jul':7, 'ago':8, 'sep':9, 'oct':10, 'nov':11, 'dic':12
}

# archivos
TABLA1_FILE = "datos_tabla1.csv"
PROCESADOS_FILE = "pdfs_procesados.txt"

In [42]:
# carpeta donde están los PDFs
pdf_folder = "../pdfs"  # CAMBIAR

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

# cargar procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

print("Total PDFs:", len(pdf_files))
print("Ya procesados:", len(procesados))

Total PDFs: 83
Ya procesados: 0


In [43]:
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print("PDFs nuevos:", len(pdfs_nuevos))

PDFs nuevos: 83


In [44]:
# lista donde guardaremos los registros válidos de la tabla 1
registros_tabla1 = []

# este patrón identifica filas con estructura:
# mes abreviado + día + mayorista + minorista
# ejemplo: mar 27 6.25 7.53
patron = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+(\d+\.\d+)\s+(\d+\.\d+)$"

# recorremos los PDFs nuevos
for pdf_name in tqdm(pdfs_nuevos, desc="Procesando PDFs"):
    
    # armamos la ruta completa del archivo PDF
    pdf_path = os.path.join(pdf_folder, pdf_name)

    try:
        # abrimos el PDF
        with pdfplumber.open(pdf_path) as pdf:
            # extraemos texto de la primera página
            # asumimos que esta tabla siempre está en la primera página
            text = pdf.pages[0].extract_text()

        # si no se pudo leer texto, saltamos al siguiente PDF
        if not text:
            continue

        # dividimos el texto en líneas
        lines = text.split("\n")

        # esta bandera indica si ya estamos dentro de la tabla 1
        leyendo_tabla1 = False

        # recorremos línea por línea
        for line in lines:
            line = line.strip().lower()

            # ------------------------------------------------------
            # 1. detectar inicio de la tabla 1
            # ------------------------------------------------------
            if "precios de huevo segun tipo de comercializacion" in line:
                leyendo_tabla1 = True
                continue

            # ------------------------------------------------------
            # 2. detectar fin de la tabla 1
            # ------------------------------------------------------
            # detenemos la lectura si encontramos señales de cierre
            if leyendo_tabla1 and (
                "var%" in line
                or "fuente: sisap mercado de productores de santa anita" in line
                or "precio del huevo de primera calidad en centro de produccion" in line
                or "oferta del huevo de primera segun macroregion" in line
            ):
                break

            # ------------------------------------------------------
            # 3. si estamos dentro de la tabla, intentamos extraer filas
            # ------------------------------------------------------
            if leyendo_tabla1:
                match = re.match(patron, line)

                if match:
                    registros_tabla1.append({
                        "pdf_name": pdf_name,
                        "mes": match.group(1),
                        "mes_num": meses_map[match.group(1)],
                        "dia": int(match.group(2)),
                        "mayorista": float(match.group(3)),
                        "minorista": float(match.group(4))
                    })

        # si todo salió bien, marcamos el PDF como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a") as f:
            f.write(pdf_name + "\n")

    except Exception as e:
        print(f"Error en {pdf_name}: {e}")

print("Registros nuevos extraídos:", len(registros_tabla1))

Procesando PDFs: 100%|██████████| 83/83 [02:39<00:00,  1.92s/it]

Registros nuevos extraídos: 963


In [ ]:
df_new = pd.DataFrame(registros_tabla1)

,pdf_name,mes,mes_num,dia,mayorista,minorista
0,09_abril_2026.pdf,mar,3,27,6.25,7.53
1,09_abril_2026.pdf,mar,3,28,6.25,7.53
2,09_abril_2026.pdf,mar,3,29,6.25,7.53
3,09_abril_2026.pdf,mar,3,30,6.18,7.45
4,09_abril_2026.pdf,mar,3,31,6.05,7.45


In [47]:
df_new.duplicated(subset=['pdf_name', 'mes_num', 'dia']).sum()

0

In [48]:
if os.path.exists(TABLA1_FILE):
    df_old = pd.read_csv(TABLA1_FILE)
else:
    df_old = pd.DataFrame()

print("Histórico:", df_old.shape)
print("Nuevos:", df_new.shape)

Histórico: (1650, 6)
Nuevos: (963, 6)


In [49]:
# unimosnuevo + historico

df_tabla1 = pd.concat([df_old, df_new], ignore_index=True)

df_tabla1.shape

(2613, 6)

In [50]:
df_tabla1 = df_tabla1.drop_duplicates(
    subset=['pdf_name', 'mes_num', 'dia', 'mayorista', 'minorista']
)

df_tabla1.shape

(1650, 6)

In [51]:
df_tabla1.to_csv(TABLA1_FILE, index=False)

print("Tabla guardada correctamente")

Tabla guardada correctamente


In [52]:
df_tabla1.sort_values(['pdf_name', 'mes_num', 'dia'])

,pdf_name,mes,mes_num,dia,mayorista,minorista
0,09_abril_2026.pdf,mar,3,27,6.25,7.53
13,09_abril_2026.pdf,mar,3,27,5.92,6.00
1,09_abril_2026.pdf,mar,3,28,6.25,7.53
14,09_abril_2026.pdf,mar,3,28,6.33,5.97
2,09_abril_2026.pdf,mar,3,29,6.25,7.53
...,...,...,...,...,...,...
1649,31_octubre_2025.pdf,oct,10,26,4.30,4.55
1644,31_octubre_2025.pdf,oct,10,27,4.73,6.06
1645,31_octubre_2025.pdf,oct,10,28,4.73,6.06
1646,31_octubre_2025.pdf,oct,10,29,4.80,6.02


In [55]:
# df_tabla1 debería ser guardado en csv para evitar correr todo el proceso
df_tabla1.to_csv("datos_tabla1.csv", index=False)

## OFERTA DEL HUEVO DE PRIMERA SEGUN MACROREGION(toneladas)


In [64]:
# mapeo de meses
meses_map = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'ago': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

# archivo final de la tabla 2
TABLA2_FILE = "datos_tabla2.csv"

# archivo para controlar qué PDFs ya fueron procesados
PROCESADOS_FILE = "pdfs_procesados_tabla2.txt"

In [65]:
# carpeta donde están los PDFs
pdf_folder = "../pdfs"   # CAMBIAR

# listar PDFs
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]

# cargar lista de PDFs ya procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r", encoding="utf-8") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

print("Total PDFs:", len(pdf_files))
print("PDFs ya procesados:", len(procesados))

Total PDFs: 83
PDFs ya procesados: 0


In [66]:
# solo nuevos
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print("PDFs nuevos por procesar:", len(pdfs_nuevos))

PDFs nuevos por procesar: 83


In [67]:
# aquí guardaremos los nuevos registros
registros_tabla2 = []

for pdf_name in tqdm(pdfs_nuevos, desc="Procesando Tabla 2"):
    pdf_path = os.path.join(pdf_folder, pdf_name)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
        
        if not text:
            continue
        
        lines = text.split("\n")
        leyendo_tabla2 = False
        
        for line in lines:
            line_lower = line.strip().lower()
            
            # empezar a leer solo cuando aparezca el título de la tabla 2
            if "oferta del huevo de primera segun macroregion" in line_lower:
                leyendo_tabla2 = True
                continue
            
            # detenerse al llegar a la fuente o a otra tabla
            if leyendo_tabla2 and (
                "fuente: empresas productoras de huevo" in line_lower
                or "precio del huevo de primera calidad en centro de produccion" in line_lower
                or "precios de huevo segun tipo de comercializacion" in line_lower
            ):
                break
            
            # dentro del bloque correcto, buscar filas tipo: mar 27 ...
            if leyendo_tabla2:
                patron_inicio = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+"
                match = re.match(patron_inicio, line_lower)
                
                if match:
                    mes = match.group(1)
                    dia = int(match.group(2))
                    
                    # quitar la parte inicial "mes día"
                    resto = re.sub(patron_inicio, "", line_lower)
                    partes = resto.split()
                    
                    valores = []
                    for x in partes:
                        x = x.replace(",", "")
                        if x in ["-", "#n/a"]:
                            valores.append(np.nan)
                        else:
                            try:
                                valores.append(float(x))
                            except:
                                pass
                    
                    # guardar solo si se detectan al menos 6 columnas numéricas
                    if len(valores) >= 6:
                        registros_tabla2.append({
                            "pdf_name": pdf_name,
                            "mes": mes,
                            "mes_num": meses_map[mes],
                            "dia": dia,
                            "oferta_lima": valores[0],
                            "oferta_sierra": valores[1],
                            "oferta_costa_norte": valores[2],
                            "oferta_selva": valores[3],
                            "oferta_costa_sur": valores[4],
                            "oferta_total": valores[5]
                        })
        
        # marcar el PDF como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a", encoding="utf-8") as f:
            f.write(pdf_name + "\n")
    
    except Exception as e:
        print(f"Error en {pdf_name}: {e}")

print("Nuevos registros extraídos:", len(registros_tabla2))

Procesando Tabla 2: 100%|██████████| 83/83 [02:39<00:00,  1.92s/it]

Nuevos registros extraídos: 1134


In [68]:
df_new = pd.DataFrame(registros_tabla2)


In [69]:
df_tabla2 = df_new.copy()

# quitar duplicados exactos si existieran
df_tabla2 = df_tabla2.drop_duplicates(
    subset=[
        "pdf_name", "mes_num", "dia",
        "oferta_lima", "oferta_sierra", "oferta_costa_norte",
        "oferta_selva", "oferta_costa_sur", "oferta_total"
    ]
)

df_tabla2.to_csv(TABLA2_FILE, index=False)

print("Tabla 2 guardada correctamente.")


Tabla 2 guardada correctamente.


In [71]:
df_tabla2

,pdf_name,mes,mes_num,dia,oferta_lima,oferta_sierra,oferta_costa_norte,oferta_selva,oferta_costa_sur,oferta_total
0,09_abril_2026.pdf,mar,3,27,156113.0,79156.0,16041.0,8718.0,0.0,260028.0
1,09_abril_2026.pdf,mar,3,28,100245.0,103792.0,7442.0,3626.0,0.0,215105.0
2,09_abril_2026.pdf,mar,3,29,35028.0,34428.0,0.0,0.0,0.0,69456.0
3,09_abril_2026.pdf,mar,3,30,198524.0,115340.0,48429.0,21317.0,0.0,383610.0
4,09_abril_2026.pdf,mar,3,31,41856.0,41125.0,0.0,4700.0,0.0,87681.0
...,...,...,...,...,...,...,...,...,...,...
1129,31_octubre_2025.pdf,oct,10,27,136600.0,123066.0,53975.0,8864.0,0.0,322505.0
1130,31_octubre_2025.pdf,oct,10,28,143049.0,41950.0,15659.0,2858.0,0.0,203516.0
1131,31_octubre_2025.pdf,oct,10,29,132777.0,110764.0,16146.0,17205.0,0.0,276892.0
1132,31_octubre_2025.pdf,oct,10,30,127747.0,80565.0,33882.0,13164.0,0.0,255357.0


In [70]:
duplicados_tabla2 = (
    df_tabla2.groupby(["pdf_name", "mes_num", "dia"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

print("Duplicados por pdf y fecha:", duplicados_tabla2.shape[0])
duplicados_tabla2.head(20)

Duplicados por pdf y fecha: 0


,pdf_name,mes_num,dia,n


Guardar resultados

In [72]:
df_tabla2 = pd.DataFrame(registros_tabla2)

# Cargar datos anteriores si existen
if os.path.exists("datos_tabla2.csv"):
    df_anterior = pd.read_csv("datos_tabla2.csv")
    df_tabla2 = pd.concat([df_anterior, df_tabla2], ignore_index=True)
    print(f" Datos anteriores: {len(df_anterior)} registros")

print(f" Total acumulado: {len(df_tabla2)} registros")



 Datos anteriores: 1134 registros
 Total acumulado: 2268 registros


In [73]:
# Guardar
df_tabla2.to_csv("datos_tabla2.csv", index=False)


## PRECIO DEL HUEVO DE PRIMERA CALIDAD EN CENTRO DE PRODUCCION (Soles por Kg)


In [ ]:


# Mapeo de meses
meses_map = {'ene':1, 'feb':2, 'mar':3, 'abr':4, 'may':5, 'jun':6,
             'jul':7, 'ago':8, 'sep':9, 'oct':10, 'nov':11, 'dic':12}

# Archivo para controlar PDFs ya procesados en Tabla 3
PROCESADOS_FILE = "pdfs_procesados_tabla3.txt"

# Cargar PDFs ya procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

registros_tabla3 = []

print(f" PDFs ya procesados (Tabla 3): {len(procesados)}")

 PDFs ya procesados (Tabla 3): 0


Procesamos solo los pdfs nuevos

In [ ]:
# Filtrar PDFs nuevos
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print(f" Total PDFs: {len(pdf_files)}")
print(f" PDFs nuevos: {len(pdfs_nuevos)}")

for pdf_name in tqdm(pdfs_nuevos, desc="Procesando Tabla 3"):
    pdf_path = os.path.join(pdf_folder, pdf_name)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
        
        lines = text.split("\n")
        
        for line in lines:
            line_lower = line.strip().lower()
            
            # Patrón: mes + día + números
            patron_inicio = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+"
            match = re.match(patron_inicio, line_lower)
            
            if match:
                mes = match.group(1)
                dia = int(match.group(2))
                
                # Extraer números
                resto = re.sub(patron_inicio, "", line_lower)
                partes = resto.split()
                
                valores = []
                for x in partes:
                    x = x.replace(",", "")
                    if x in ["-", "#n/a"]:
                        valores.append(np.nan)
                    else:
                        try:
                            valores.append(float(x))
                        except:
                            pass
                
                # Asignar según cantidad de valores (Ica, La Libertad, Lima)
                if len(valores) >= 3:
                    precio_ica = valores[0]
                    precio_la_libertad = valores[1]
                    precio_lima = valores[2]
                elif len(valores) == 2:
                    precio_ica = valores[0]
                    precio_la_libertad = np.nan
                    precio_lima = valores[1]
                elif len(valores) == 1:
                    precio_ica = valores[0]
                    precio_la_libertad = np.nan
                    precio_lima = np.nan
                else:
                    continue
                
                registros_tabla3.append({
                    "pdf_name": pdf_name,
                    "mes": mes,
                    "mes_num": meses_map[mes],
                    "dia": dia,
                    "precio_ica": precio_ica,
                    "precio_la_libertad": precio_la_libertad,
                    "precio_lima": precio_lima
                })
        
        # Marcar como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a") as f:
            f.write(pdf_name + "\n")
            
    except Exception as e:
        print(f" Error en {pdf_name}: {e}")

print(f" Nuevos registros Tabla 3: {len(registros_tabla3)}")

 Total PDFs: 83
 PDFs nuevos: 83


Procesando Tabla 3: 100%|██████████| 83/83 [03:13<00:00,  2.33s/it]

 Nuevos registros Tabla 3: 3566


guardamos en csv

In [ ]:
df_tabla3 = pd.DataFrame(registros_tabla3)

# Cargar datos anteriores si existen
if os.path.exists("datos_tabla3.csv"):
    df_anterior = pd.read_csv("datos_tabla3.csv")
    df_tabla3 = pd.concat([df_anterior, df_tabla3], ignore_index=True)
    print(f"📚 Datos anteriores: {len(df_anterior)} registros")

print(f" Total acumulado Tabla 3: {len(df_tabla3)} registros")




 Total acumulado Tabla 3: 3566 registros


In [ ]:
# Guardar
df_tabla3.to_csv("datos_tabla3.csv", index=False)